# 2. How do you evaluate a RAG chatbot?

In the malnutrition and TB lessons, evaluation meant comparing predictions with **labels**: who really developed
GAM, who really had TB. Count the hits and misses and you have sensitivity and ROC-AUC.

Our chatbot outputs a paragraph. What is the "label" for a paragraph?

| Option | How | Problem |
|---|---|---|
| **Expert reference answers** | a clinician writes the right answer to each test question; compare | slow and costly to write; one question can have many right answers; and **who checked the reference?** An unvalidated reference just moves the error |
| **Expert review of every answer** | a clinician reads each answer and grades it | the gold standard, but too slow to repeat every time we change a setting |
| **An LLM as judge, no reference** | a second, stronger LLM checks each answer against the question and the passages it came from | fast and repeatable; checked once against people before a real decision |

We use the third one every time we change something, and keep expert review for the final decision.

## Three questions a judge can answer without knowing the right answer

A RAG answer can go wrong at two places: the **search** (wrong passages) or the **model** (right passages, bad
answer). Three checks, known as the **RAG triad**, separate these, and none of them needs a reference answer:

| Check | The judge reads | It asks | A low score means |
|---|---|---|---|
| **Context relevance** | question + retrieved passages | Are these passages about the question? | **search** failed: the model never saw the right page |
| **Groundedness** (also called faithfulness) | answer + retrieved passages | Is what the answer says supported by the passages? | the **model** added something the guideline doesn't say: a hallucination |
| **Answer relevancy** | question + answer | Does the answer address what was asked? | the **model** dodged, drifted or answered a different question |

```
             context relevance                 groundedness
  question ─────────────────────── passages ─────────────────── answer
      └──────────────────────── answer relevancy ─────────────────┘
```

For clinical decision support, **groundedness** matters most: an ungrounded answer is a confident statement the
guideline doesn't support.

What none of the three can tell you: whether the **guideline itself** is right, or whether the answer left out
something important that was in a passage the search didn't find. That still needs people.

**Ragas** is an open-source library that implements these checks: each is a carefully written prompt for the judge,
plus code that turns the judge's replies into a score from 0 to 1.

In [1]:
import os
import pandas as pd

import rag
import rag_eval

pd.set_option("display.width", 200, "display.max_colwidth", None, "display.max_rows", 40)   # show full texts
assert os.environ.get("OPENROUTER_API_KEY"), "No key yet: open llm-cds/.env, paste the key after OPENROUTER_API_KEY=, save, restart the kernel"
print("assistant:", rag.Settings().model, "  judge:", rag_eval.JUDGE_MODEL)   # the judge: a different, stronger model

assistant: google/gemma-4-26b-a4b-it   judge: anthropic/claude-haiku-5.5


## The test questions

24 questions written for this workshop: 20 that the guidelines answer (single facts, doses, tables, questions
needing two passages, one in Amharic) and 4 they don't (to check that the assistant refuses).

The file also has reference answers. **We don't use them**: nobody has validated them, and the point of this
notebook is to evaluate without them.

In [2]:
questions = rag_eval.load_questions()
print(questions.type.value_counts().to_string(), "\n")
questions

type
answerable      20
out_of_scope     4 



,id,type,question
0,chol_01,answerable,"What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported?"
1,chol_02,answerable,"A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, at what dose, and what can I use instead if that drug isn't suitable?"
2,chol_03,answerable,A 3-year-old comes to the CTC severely dehydrated from cholera. How should I give the IV Ringer's lactate — how much and over what time?
3,chol_04,answerable,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far apart should the two doses be given, and from what age can people be vaccinated?"
4,nutr_01,answerable,Our emergency nutrition survey found an acute malnutrition rate of 12% with no aggravating factors. Which food and nutrition interventions do the surveillance benchmarks call for?
5,nutr_02,answerable,"In an emergency vitamin A supplementation round, what preventive dose and how often should infants aged 6–12 months and children over 1 year receive?"
6,nutr_03,answerable,"If a displaced population is relying completely on general food distribution, what is the minimum energy the ration has to provide per person?"
7,csd_01,answerable,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?
8,csd_02,answerable,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C, and below what temperatures does this development fail to complete?"
9,csd_03,answerable,"In the climate-sensitive disease case definitions, when should a patient be classified as a suspected case of meningococcal meningitis?"


## Step 1. What does the search find?

Before any LLM: run only the search (the setup we built in notebook 1: PDF text, 200-word windows, keywords), and
look at which pages come back for each question.

In [3]:
baseline = rag.Settings()                      # docs="pdf", chunking="fixed", retriever="tfidf", k=5
retriever = rag.build(baseline)


def top_sources(question, k=5):
    hits = retriever.search(question, k)
    return "; ".join(rag.source_ids(hits))


questions.assign(sources=questions.question.map(top_sources))[["id", "question", "sources"]]

,id,question,sources
0,chol_01,"What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported?",ephi_cholera_2022:p14; ephi_cholera_2022:p71; ephi_cholera_2022:p16; ephi_cholera_2022:p87; ephi_cholera_2022:p15
1,chol_02,"A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, at what dose, and what can I use instead if that drug isn't suitable?",ephi_cholera_2022:p25; ephi_cholera_2022:p38; ephi_cholera_2022:p70; ephi_cholera_2022:p25; ephi_cholera_2022:p34
2,chol_03,A 3-year-old comes to the CTC severely dehydrated from cholera. How should I give the IV Ringer's lactate — how much and over what time?,ephi_cholera_2022:p28; ephi_cholera_2022:p29; ephi_cholera_2022:p25; ephi_cholera_2022:p32; ephi_cholera_2022:p28
3,chol_04,"For a reactive oral cholera vaccination campaign with the stockpile vaccines, how far apart should the two doses be given, and from what age can people be vaccinated?",ephi_cholera_2022:p70; ephi_cholera_2022:p69; ephi_cholera_2022:p69; ephi_cholera_2022:p70; ephi_cebs_2021:p26
4,nutr_01,Our emergency nutrition survey found an acute malnutrition rate of 12% with no aggravating factors. Which food and nutrition interventions do the surveillance benchmarks call for?,ephi_emergency_nutrition_2022:p30; ephi_emergency_nutrition_2022:p48; ephi_emergency_nutrition_2022:p45; ephi_emergency_nutrition_2022:p4; ephi_emergency_nutrition_2022:p40
5,nutr_02,"In an emergency vitamin A supplementation round, what preventive dose and how often should infants aged 6–12 months and children over 1 year receive?",ephi_emergency_nutrition_2022:p52; ephi_emergency_nutrition_2022:p52; ephi_emergency_nutrition_2022:p84; ephi_emergency_nutrition_2022:p3; moh_acute_malnutrition_2019:p19
6,nutr_03,"If a displaced population is relying completely on general food distribution, what is the minimum energy the ration has to provide per person?",ephi_emergency_nutrition_2022:p39; ephi_emergency_nutrition_2022:p38; ephi_emergency_nutrition_2022:p46; ephi_emergency_nutrition_2022:p37; ephi_cholera_2022:p4
7,csd_01,Which diseases are prioritized for climate-sensitive disease surveillance in Ethiopia?,ephi_climate_sensitive_diseases_2022:p55; ephi_climate_sensitive_diseases_2022:p7; ephi_climate_sensitive_diseases_2022:p54; ephi_climate_sensitive_diseases_2022:p3; ephi_climate_sensitive_diseases_2022:p52
8,csd_02,"How long does the malaria parasite need to develop inside the mosquito at about 25 °C, and below what temperatures does this development fail to complete?",ephi_climate_sensitive_diseases_2022:p42; ephi_climate_sensitive_diseases_2022:p57; ephi_climate_sensitive_diseases_2022:p55; ephi_cebs_2021:p24; moh_tb_2017:p165
9,csd_03,"In the climate-sensitive disease case definitions, when should a patient be classified as a suspected case of meningococcal meningitis?",ephi_climate_sensitive_diseases_2022:p54; ephi_climate_sensitive_diseases_2022:p55; ephi_climate_sensitive_diseases_2022:p7; ephi_climate_sensitive_diseases_2022:p59; ephi_climate_sensitive_diseases_2022:p3


> **PAUSE (3 min).** Pick three questions. Do the sources look right (the cholera guideline for a cholera
> question, the SAM guideline for a malnutrition question)? Which questions got sources from the wrong guideline?
> What does the search return for the out-of-scope questions?

Looking at 24 rows by eye is fine once. For 7 setups, every time something changes, we need the judge.

## Step 2. One question, judged by hand

Answer one question and run each of the three checks on it, to see what the judge does.

In [4]:
question = questions.question.iloc[0]
result = rag.answer(question, retriever, baseline)
passages = result["contexts"]                   # the texts of the 5 passages the search returned
print(question, "\n")
print(result["answer"])

What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported? 

A cholera alert (suspected cholera outbreak) is defined by finding at least one of the following: A) Two or more people aged 2 years or older with acute watery diarrhea and severe dehydration, or dying from acute watery diarrhea, from the same area within 1 week of one another; or B) One case of acute watery diarrhea [1]. The guidelines do not specify a required timeframe for how quickly these findings must be reported.


In [5]:
from ragas.metrics.collections import AnswerRelevancy, ContextRelevance, ResponseGroundedness

judge = rag_eval.judge()
checks = {
    "context relevance": (ContextRelevance(llm=judge), dict(user_input=question, retrieved_contexts=passages)),
    "groundedness":      (ResponseGroundedness(llm=judge), dict(response=result["answer"], retrieved_contexts=passages)),
    "answer relevancy":  (AnswerRelevancy(llm=judge, embeddings=rag_eval.embeddings()),
                          dict(user_input=question, response=result["answer"])),
}
for name, (metric, inputs) in checks.items():
    score = await metric.ascore(**inputs)
    print(f"{name:18} {score.value:.2f}")

context relevance  1.00


groundedness       0.50


answer relevancy   0.84


What happens inside, roughly:

- **Context relevance:** the judge rates, twice, whether the passages are relevant to the question (0, 1 or 2);
  the two ratings are averaged and scaled to 0–1.
- **Groundedness:** the judge rates, twice, whether the answer is supported by the passages: fully (2), partly (1)
  or not (0); averaged and scaled to 0–1. An answer with one invented dose among correct ones gets about 0.5.
- **Answer relevancy:** the judge writes the questions that this answer *would* be an answer to; Ragas measures how
  close in meaning they are to the real question (with an embedding model). It also scores 0 if the answer is
  non-committal ("I'm not sure").

## Step 3. All 24 questions

`rag_eval.evaluate_setup` does the same for every question, 8 at a time, and records everything in **MLflow**:
one run for this setup, with its settings and prompt, and one **trace** per question showing the search, the LLM
call and the three scores. About a minute.

In [6]:
base = rag_eval.evaluate_setup(baseline, "1. baseline: PDF text, fixed, keywords")

2026/10/28 21:42:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/28 21:42:34 INFO mlflow.store.db.utils: Updating database tables


2026/10/28 21:42:35 INFO mlflow.tracking.fluent: Experiment with name 'guideline-assistant' does not exist. Creating a new experiment.


2026/10/28 21:42:36 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/24 [Elapsed: 00:00, Remaining: ?]

...

Evaluating:  88%|████████▊ | 21/24 [Elapsed: 00:25, Remaining: 00:03]

Evaluating:  96%|█████████▌| 23/24 [Elapsed: 00:26, Remaining: 00:01]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 28%, scorers: 72%]

Evaluating: 100%|██████████| 24/24 [Elapsed: 00:26, Remaining: 00:00] [predict_fn: 28%, scorers: 72%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: 1. baseline: PDF text, fixed, keywords
  Run ID: 0972dd7e786a470a857be7a53eff51d6

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.

1. baseline: PDF text, fixed, keywords: context_relevance 0.96, groundedness 0.97, answer_relevancy 0.85, refused_in_scope 3, refused_out_of_scope 4   (MLflow run 0972dd7e)


The last line printed has five numbers: the three scores, averaged over the questions the assistant **answered**,
and two counts of **refusals** (answers that are exactly "The guidelines provided do not cover this."):

- `refused_out_of_scope`: refusals to the 4 questions the guidelines don't cover. **Correct**: should be 4.
- `refused_in_scope`: refusals to questions the guidelines *do* answer. **Wrong**: each one is a missed answer;
  should be 0.

**Why refusals are counted apart.** For "what insulin dose for a child with type 1 diabetes?" the right behaviour is
the refusal. But a refusal says nothing that the passages could support, and doesn't address the question
(answer relevancy 0): the judge would punish honesty.

Every refusal, one by one:## Step 4. Read the worst answers

Averages hide the failures. Sort by the scores, then read the question, the answer and the passages, and decide
for yourself whether the judge is right.

In [7]:
refusals = base[base.refused][["id", "type", "question", "sources"]]
refusals.insert(2, "verdict", refusals.type.map({"out_of_scope": "correct: the guidelines don't cover it",
                                                 "answerable": "WRONG: the guidelines answer this"}))
print(f"{(refusals.type == 'out_of_scope').sum()} correct refusals (of 4), "
      f"{(refusals.type == 'answerable').sum()} wrong refusals (of 20)")
refusals

4 correct refusals (of 4), 3 wrong refusals (of 20)


,id,type,verdict,question,sources
9,csd_03,answerable,WRONG: the guidelines answer this,"In the climate-sensitive disease case definitions, when should a patient be classified as a suspected case of meningococcal meningitis?",ephi_climate_sensitive_diseases_2022:p54; ephi_climate_sensitive_diseases_2022:p55; ephi_climate_sensitive_diseases_2022:p7; ephi_climate_sensitive_diseases_2022:p59; ephi_climate_sensitive_diseases_2022:p3
15,sam_02,answerable,WRONG: the guidelines answer this,"Which routine antibiotic should a child with severe acute malnutrition but no medical complications receive, at what dose and for how long?",moh_acute_malnutrition_2019:p36; moh_acute_malnutrition_2019:p13; moh_acute_malnutrition_2019:p7; ephi_emergency_nutrition_2022:p7; moh_acute_malnutrition_2019:p25
17,tb_01,answerable,WRONG: the guidelines answer this,"What isoniazid dose should I prescribe for TB preventive therapy in adults and in children, for how long, and should anything be given with it?",moh_tb_2017:p57; moh_tb_2017:p57; moh_tb_2017:p59; moh_tb_2017:p58; moh_tb_2017:p57
20,oos_01,out_of_scope,correct: the guidelines don't cover it,What is the first-line drug and adult dose for uncomplicated P. falciparum malaria — should I use artemether-lumefantrine?,ephi_climate_sensitive_diseases_2022:p57; ephi_climate_sensitive_diseases_2022:p57; moh_tb_2017:p43; moh_tb_2017:p44; moh_tb_2017:p182
21,oos_02,out_of_scope,correct: the guidelines don't cover it,"How many vials of antivenom should be given for a venomous snakebite, and how is it administered?",moh_tb_2017:p57; moh_tb_2017:p59; moh_tb_2017:p58; moh_tb_2017:p18; moh_tb_2017:p19
22,oos_03,out_of_scope,correct: the guidelines don't cover it,At what ages should infants receive the rotavirus vaccine doses to help prevent diarrhoea?,moh_acute_malnutrition_2019:p41; moh_acute_malnutrition_2019:p82; ephi_emergency_nutrition_2022:p15; moh_tb_2017:p92; moh_tb_2017:p170
23,oos_04,out_of_scope,correct: the guidelines don't cover it,Someone was bitten by a suspected rabid dog — what is the post-exposure rabies vaccine schedule?,ephi_cebs_2021:p14; ephi_cebs_2021:p13; moh_acute_malnutrition_2019:p41; ephi_cholera_2022:p61; ephi_climate_sensitive_diseases_2022:p60


> **PAUSE (3 min).** For a wrong refusal, look at its sources: did the search find the right guideline and page?
> If not, the model was right to refuse **given what it was shown**: the failure is in the search.

## Step 4. Read the worst answers

Averages hide the failures. Sort by the scores, then read the question, the answer and the passages, and decide
for yourself whether the judge is right.

The table has one row per question, lowest groundedness first. A refusal (`refused` True) has no score that means
anything, so look at the answered questions.

In [8]:
scores = base[["id", "question", "refused"] + rag_eval.METRICS].copy()
scores[rag_eval.METRICS] = scores[rag_eval.METRICS].astype(float).round(2)
scores.sort_values(["groundedness", "context_relevance"])

,id,question,refused,context_relevance,groundedness,answer_relevancy
20,oos_01,What is the first-line drug and adult dose for uncomplicated P. falciparum malaria — should I use artemether-lumefantrine?,True,0.00,0.0,0.51
21,oos_02,"How many vials of antivenom should be given for a venomous snakebite, and how is it administered?",True,0.00,0.0,0.49
22,oos_03,At what ages should infants receive the rotavirus vaccine doses to help prevent diarrhoea?,True,0.00,0.0,0.52
9,csd_03,"In the climate-sensitive disease case definitions, when should a patient be classified as a suspected case of meningococcal meningitis?",True,0.50,0.0,0.54
15,sam_02,"Which routine antibiotic should a child with severe acute malnutrition but no medical complications receive, at what dose and for how long?",True,0.50,0.0,0.48
23,oos_04,Someone was bitten by a suspected rabid dog — what is the post-exposure rabies vaccine schedule?,True,0.50,0.0,0.51
17,tb_01,"What isoniazid dose should I prescribe for TB preventive therapy in adults and in children, for how long, and should anything be given with it?",True,1.00,0.0,0.53
0,chol_01,"What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported?",False,1.00,0.5,0.85
14,sam_01,"A child in the outpatient therapeutic programme still hasn't met the cure criteria. After how long is she classed as a non-responder, and when would a child instead be counted as a defaulter?",False,0.50,1.0,0.83
18,tb_02,"For drug-susceptible TB, when do I record a treatment outcome as 'treatment failure' versus 'lost to follow-up'?",False,0.75,1.0,0.85


`explain` prints one question in full: the question, the answer, its three scores, and the start of each of the 5
passages the search gave the model. The cell below uses it on the **answered question with the lowest groundedness**
(the answer least supported by its passages); its id is printed first. Which question that is can change from run
to run (the model and the judge vary a little). In the run saved in this notebook it is `chol_01`, the cholera
alert: read the answer against passage [1] and decide what the model got wrong. In other runs it was `nutr_01`,
where the PDF text scrambles the table on page 30 and the model misreads which row applies.

In [9]:
def explain(table, question_id, chars=600):
    """The question, the answer, the scores, and the start of each passage the answer was based on."""
    row = table.set_index("id").loc[question_id]
    print(row.question, "\n\nANSWER:", row.answer, "\n")
    print("  ".join(f"{m} {row[m]:.2f}" if pd.notna(row[m]) else f"{m} -" for m in rag_eval.METRICS), row.judge_notes)
    for n, (source, text) in enumerate(zip(row.sources.split("; "), row.passages), 1):
        print(f"\n[{n}] {source}: {text[:chars]}...")


answered = scores[~scores.refused]
worst = answered.sort_values(["groundedness", "context_relevance"]).id.iloc[0]
print(f"The answered question with the lowest groundedness: {worst}\n")
explain(base, worst)

The answered question with the lowest groundedness: chol_01

What findings should make me raise a cholera alert (suspected outbreak) in my area, and how quickly does it have to be reported? 

ANSWER: A cholera alert (suspected cholera outbreak) is defined by finding at least one of the following: A) Two or more people aged 2 years or older with acute watery diarrhea and severe dehydration, or dying from acute watery diarrhea, from the same area within 1 week of one another; or B) One case of acute watery diarrhea [1]. The guidelines do not specify a required timeframe for how quickly the alert must be reported [1][4]. 

context_relevance 1.00  groundedness 0.50  answer_relevancy 0.85 

[1] ephi_cholera_2022:p14: 7 Guideline for Cholera Surveillance and Outbreak Response Ethiopia Detection and Diagnosis Cholera cases are detected based on clinical suspicion in patients who present with severe acute watery diarrhea. The suspicion is then diagnosed by identifying V . cholerae in stool sam

> **Exercise (10 min), in pairs.** Take the three answers with the lowest scores (any of the three checks).
> For each, decide:
> 1. Was it the **search** (wrong passages: low context relevance) or the **model** (good passages, but low
>    groundedness or answer relevancy)?
> 2. Do you agree with the judge? Read the passages (`explain(base, "csd_03")`, with the question's id) and check.
>
> Are there any **refusals to in-scope questions**? Look at their sources: did the search find the right page?

## Step 5. The same, in the MLflow UI

In a terminal, from `llm-cds/`, environment active:

```bash
mlflow ui            # http://127.0.0.1:5000
```

Experiment **guideline-assistant** → the run **1. baseline...** → the **Traces** (or Evaluations) tab. Click one
question: the trace shows the `search` step (the passages, with their pages and scores), the `llm` step (the exact
messages and the answer), and the three scores on the right. The run's **Parameters**
record the full setup (model, prompt version, chunking, k, judge) and a fingerprint of the passages: everything
needed to reproduce this score.

## What these three checks don't cover

> **PAUSE (5 min), discussion.**
> - The judge sees only the passages the search returned. Can it notice an answer that is grounded in those
>   passages but **incomplete**, because the important line was on a page the search missed?
> - Before this assistant reaches a health worker, what would you still want a clinician to review, and how many
>   answers?

The checks tell us whether the assistant answered **from the guideline**. Whether the guideline answer is the
**complete and right** clinical answer still needs a clinician: grade a sample of answers once, and check the
judge agrees with them, before the assistant is used for real decisions.

## Your turn

Write two questions of your own: one you expect the guidelines to answer well, and one you expect to be hard
(a table, an everyday phrasing, Amharic, or something the guidelines may not cover). Evaluate only those, then
read the scores: do they match what you expected?

In [ ]:
my_questions = [
    "",     # your first question, between the quotes
    "",     # your second question
]
assert all(q.strip() for q in my_questions), "Write your two questions between the quotes first"

mine = pd.DataFrame({"id": [f"my_{i}" for i in range(1, len(my_questions) + 1)], "type": "mine",
                     "question": my_questions})
my_scores = rag_eval.evaluate_setup(baseline, "my questions", mine)
my_scores[["question", "sources", "answer"] + rag_eval.METRICS]